# 02 · Chạy thử vLLM với một mô hình 7B (Tuần 1 · TV2)

**Chạy ở đâu:** Google Colab hoặc Kaggle với GPU T4 (16 GB). vLLM không chạy trên Windows, và GPU 6 GB của máy cá nhân không đủ cho mô hình 7B.

**Mục tiêu:** xác nhận (1) vLLM phục vụ được một mô hình 7B lượng tử hóa AWQ 4-bit trên GPU miễn phí, (2) `rag/llm_client.py` gọi được qua API tương thích OpenAI và lưu đệm JSONL, (3) ghi lại VRAM và tốc độ sinh để ước lượng thời gian cho Tuần 4–5.

Chưa ai chạy notebook này: các ô đầu ra đang trống. Sau khi chạy, điền số vào `docs/research_log.md`.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total,memory.used --format=csv

In [ ]:
%%capture
!pip install vllm
!git clone https://github.com/yuhna111/NEXUS2026-FeelTheBeat repo
!pip install -e repo

In [ ]:
import subprocess, time, requests

MODEL = "Qwen/Qwen2.5-7B-Instruct-AWQ"   # ghi đúng mã này vào configs/models.yaml nếu chọn
log = open("vllm.log", "w")
server = subprocess.Popen(
    ["vllm", "serve", MODEL, "--port", "8000",
     "--max-model-len", "4096", "--gpu-memory-utilization", "0.90"],
    stdout=log, stderr=subprocess.STDOUT,
)

t0 = time.time()
while True:
    if server.poll() is not None:
        raise RuntimeError("vLLM dừng sớm — xem vllm.log:\n" + open("vllm.log").read()[-3000:])
    try:
        if requests.get("http://localhost:8000/v1/models", timeout=2).ok:
            break
    except requests.RequestException:
        pass
    time.sleep(5)
print(f"Máy chủ sẵn sàng sau {time.time() - t0:.0f} giây")

In [ ]:
!nvidia-smi --query-gpu=memory.used,memory.total --format=csv

In [ ]:
import os, sys, time
os.environ["VLLM_BASE_URL"] = "http://localhost:8000/v1"
os.environ["LLM_CACHE_DIR"] = "llm_cache"
sys.path.insert(0, "repo")
from rag.llm_client import LLMClient

client = LLMClient(provider="vllm", model=MODEL)
system = "Bạn là trợ lý trả lời câu hỏi. Trả lời ngắn gọn, chính xác bằng tiếng Việt."
questions = [
    "Việt Nam hiện có bao nhiêu tỉnh, thành phố trực thuộc trung ương?",
    "Tỉnh Lâm Đồng mới được hợp nhất từ những tỉnh nào?",
    "Chủ tịch UBND huyện Gia Lâm hiện nay đóng trụ sở ở đâu?",
]
for q in questions:
    t = time.time()
    rec = client.generate(q, system=system)
    dt = time.time() - t
    out_tok = rec["usage"].get("completion_tokens", 0)
    print(f"Q: {q}\nA: {rec['output']}\n   {dt:.1f} s · {out_tok} token ra · {out_tok / dt:.1f} token/s\n")

In [ ]:
# Gọi lại lần hai phải lấy từ bộ đệm (gần như tức thì, không tốn GPU).
t = time.time()
client.generate(questions[0], system=system)
print(f"Lần hai: {time.time() - t:.3f} s")
!wc -l llm_cache/*.jsonl

In [ ]:
server.terminate()